In [12]:
# pip install datasets

In [8]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

In [9]:
# 1) Identify emotion columns safely
df = pd.read_csv("goemotions_cleaned.csv")
emotion_cols = [c for c in df.columns if c != "text"]   # or explicitly list them

df_train, df_temp = train_test_split(df, test_size=0.3, random_state=42, shuffle=True) #splits df_train to 70%, df_temp to 30%
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, shuffle=True) #splits df_temp into df_val and df_test into 15% respectively

In [10]:
# 2) Build HF datasets
train_ds = Dataset.from_pandas(df_train.reset_index(drop=True))
val_ds   = Dataset.from_pandas(df_val.reset_index(drop=True))
test_ds  = Dataset.from_pandas(df_test.reset_index(drop=True))

In [24]:
# Candidate columns excluding text
candidates = [c for c in df_train.columns if c != "text"]

bad = []
good = []

for c in candidates:
    s = df_train[c].dropna()
    # treat as label column if values are subset of {0,1} (or {0.0,1.0})
    vals = set(s.unique().tolist())
    if vals.issubset({0, 1, 0.0, 1.0, True, False}):
        good.append(c)
    else:
        bad.append((c, list(vals)[:10]))  # show sample values

print("✅ label columns:", len(good))
print("❌ non-label columns (sample values):")
for c, v in bad[:20]:
    print(c, "->", v)

✅ label columns: 29
❌ non-label columns (sample values):
id -> ['ee6xg8z', 'ed9sv72', 'ed7qrmu', 'eerrpwy', 'ef37id3', 'ee8d7tz', 'eew60b1', 'eeg2hmk', 'eczf23k', 'eczopuk']
author -> ['The_DilDonald', 'D4rk_unicorn', 'runninhillbilly', 'ItsokayIknowIamright', 'magnolias_n_peonies', 'aanarchist', 'cisxuzuul', 'wordswords321', 'mrnacho69', 'just_a_random_dood']
subreddit -> ['medicine', 'facepalm', 'TopMindsOfReddit', 'TIHI', 'heroesofthestorm', 'Habs', 'riverdale', 'BPDlovedones', 'leafs', 'The_Donald']
link_id -> ['t3_ab9bcw', 't3_ad4ixl', 't3_abm34g', 't3_acl0qw', 't3_ac2ucn', 't3_adlowj', 't3_aiuzcv', 't3_agrpvf', 't3_ainjfd', 't3_ak45kz']
parent_id -> ['t3_ab9bcw', 't1_ed60j11', 't3_acl0qw', 't3_ac2ucn', 't1_efetsqm', 't1_ee9or56', 't3_adlowj', 't3_aiuzcv', 't3_ainjfd', 't3_ak45kz']
created_utc -> [1547829250.0, 1548353539.0, 1547567113.0, 1546911754.0, 1546649612.0, 1547829261.0, 1548615694.0, 1547042833.0, 1547173906.0, 1546649625.0]
rater_id -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [25]:
emotion_cols = good   # <-- use only true 0/1 label columns
print(emotion_cols[:10], "...")  # quick sanity check

non_binary = []
for c in emotion_cols:
    vals = set(df_train[c].dropna().unique().tolist())
    if not vals.issubset({0,1,0.0,1.0,True,False}):
        non_binary.append((c, vals))
non_binary[:10]

['example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire'] ...


[]

In [26]:
for df_ in (df_train, df_val, df_test):
    df_[emotion_cols] = (
        df_[emotion_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .astype(np.float32)
    )

In [23]:
# 3) Tokeniser + preprocess
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def preprocess(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    labels = np.array([batch[c] for c in emotion_cols], dtype=np.float32).T
    enc["labels"] = labels.tolist()
    return enc

remove_cols = ["text"] + emotion_cols
train_ds = train_ds.map(preprocess, batched=True, remove_columns=remove_cols)
val_ds   = val_ds.map(preprocess, batched=True, remove_columns=remove_cols)
test_ds  = test_ds.map(preprocess, batched=True, remove_columns=remove_cols)

Map: 100%|██████████| 8701/8701 [00:00<00:00, 10845.79 examples/s]


In [30]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [31]:
# 4) Model configured for multi-label
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(emotion_cols),
    problem_type="multi_label_classification",
)

# 5) Metrics with thresholding
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))           # sigmoid
    y_pred = (probs >= 0.5).astype(int)         # later: tune per-class thresholds

    return {
        "micro_f1": f1_score(labels, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(labels, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, y_pred, average="micro", zero_division=0),
    }

args = TrainingArguments(
    output_dir="roberta_goemotions",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate(test_ds)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 448.57it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 